In [ ]:
from google.colab import files

uploaded = files.upload()  # 📌 Opens a file upload window in Google Colab


Saving mixtec-train.txt to mixtec-train.txt
Saving mixtec-val.txt to mixtec-val.txt
Saving spanish-train.txt to spanish-train.txt
Saving spanish-val.txt to spanish-val.txt


In [ ]:
# 📌 Install necessary libraries
!pip install datasets transformers huggingface_hub --upgrade

import pandas as pd
import json
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, TrainingArguments, Trainer, AutoModelForSeq2SeqLM
import wandb
import torch

# 📌 Read training files
with open("mixtec-train.txt", "r", encoding="utf-8") as f_mix_train, open("spanish-train.txt", "r", encoding="utf-8") as f_esp_train:
    mixtec_train = f_mix_train.readlines()
    spanish_train = f_esp_train.readlines()

assert len(mixtec_train) == len(spanish_train), "⚠️ Error: Training files have different line counts."

# 📌 Read validation files
with open("mixtec-val.txt", "r", encoding="utf-8") as f_mix_val, open("spanish-val.txt", "r", encoding="utf-8") as f_esp_val:
    mixtec_val = f_mix_val.readlines()
    spanish_val = f_esp_val.readlines()

assert len(mixtec_val) == len(spanish_val), "⚠️ Error: Validation files have different line counts."

# 📌 Create DataFrames for training and validation
train_df = pd.DataFrame({"mixtec": [m.strip() for m in mixtec_train], "spanish": [s.strip() for s in spanish_train]})
val_df = pd.DataFrame({"mixtec": [m.strip() for m in mixtec_val], "spanish": [s.strip() for s in spanish_val]})

# 📌 Convert DataFrames to Hugging Face dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# 📌 Create DatasetDict structure
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

# 📌 Load the mBART-50 tokenizer
model_name = "facebook/mbart-large-50"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 📌 Set source and target languages
tokenizer.src_lang = "mix_Latn"  # 📌 Mixtec as input language
tokenizer.tgt_lang = "es_XX"     # 📌 Spanish as output language

# 📌 Preprocessing function
def preprocess_function(examples):
    inputs = examples["mixtec"]   # 📌 Input is now Mixtec
    targets = examples["spanish"] # 📌 Output is now Spanish

    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=128)
    labels = tokenizer(targets, padding="max_length", truncation=True, max_length=128)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 📌 Tokenize dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 📌 Load the mBART-50 model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 📌 Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔍 Using device: {device}")
model.to(device)

# 📌 Training configuration
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,  # 📌 Adjust based on available memory
    per_device_eval_batch_size=8,
    num_train_epochs=5,  # 📌 More epochs for limited data
    logging_dir="./logs",
    logging_steps=50,  # 📌 Reduce logging frequency for stability
    learning_rate=3e-5,  # 📌 Fine-tune learning rate
    weight_decay=0.01,  # 📌 Regularization to avoid overfitting
    warmup_ratio=0.06,  # 📌 6% warmup during training
    fp16=True,  # 📌 Enable FP16 if using GPU
    report_to="wandb",  # 📌 Report metrics to Weights & Biases
    push_to_hub=False
)

# 📌 Initialize Weights & Biases
wandb.login()
wandb.init(project="mbart_finetuning_mixteco_espanol", name="mbart_train_run")

# 📌 Instantiate Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
)

# 📌 Start training
trainer.train()

# 📌 Save the fine-tuned model
model.save_pretrained("./mbart_modelo_mixteco_espanol")
tokenizer.save_pretrained("./mbart_modelo_mixteco_espanol")

print("✅ Training completed and model saved.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.0/468.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.28.1
    Uninstalling huggingface-hub-0.28.1:
      Successfully uninstalled huggingface-hub-0.28.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.48.3
    Uninstalling transformers-4.48.3:
      Successfully uninstalled transformers-4.48.3


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hsantiago13 (hsantiago13-university-aut-noma-de-quer-taro) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


<ipython-input-3-f2209ddcfadd>:92: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss
1,3.197600,0.558517
2,0.502100,0.186174
3,0.449900,0.167369
4,0.370800,0.174164
5,0.312500,0.181859


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2810: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


✅ Entrenamiento completado y modelo guardado.


In [ ]:
# Descargar modelo en Colab
#from google.colab import files
#import shutil

# shutil.make_archive("modelo_mixteco_espanol", 'zip', "./modelo_mixteco_espanol")
# files.download("modelo_mixteco_espanol.zip")

NameError: name 'model' is not defined

In [ ]:
import torch
import sacrebleu
import evaluate

# 📌 Function to generate translations using the trained model
def generate_translations(model, tokenizer, dataset, max_length=128):
    # Predict translations using the trained model
    predictions = []
    for example in dataset:
        # Tokenize the input (Mixtec text) and move it to the appropriate device
        inputs = tokenizer(example["mixtec"], return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():  # Disable gradient calculation for inference
            generated_ids = model.generate(inputs['input_ids'], max_length=max_length)
        # Decode the generated token IDs into text
        translated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        predictions.append(translated_text)
    return predictions

# 📌 Generate predictions for the validation dataset
predictions = generate_translations(model, tokenizer, val_dataset)

# 📌 Function to calculate BLEU score
def calculate_bleu(predictions, references):
    bleu = sacrebleu.corpus_bleu(predictions, [references])  # Compute BLEU score using SacreBLEU
    return bleu

# 📌 Function to calculate TER (Translation Edit Rate) using `evaluate`
def calculate_ter(predictions, references):
    ter_metric = evaluate.load("ter")  # Load the TER metric from `evaluate` library
    ter_score = ter_metric.compute(predictions=predictions, references=references)
    return ter_score

# 📌 Get reference translations (Spanish translations)
references = val_df["spanish"].tolist()

# 📌 Compute the evaluation metrics
bleu_score = calculate_bleu(predictions, references)
ter_score = calculate_ter(predictions, references)

# 📌 Display the evaluation results
print(f"BLEU: {bleu_score.score:.2f}")
print(f"TER: {ter_score['score']:.2f}")

BLEU: 2.87
TER: 111.26
